In [11]:
import pandas as pd
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [12]:
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'attack', 'level'
]

data_train = pd.read_csv("NSLKDD/KDDTrain+.txt", names=columns)
data_test = pd.read_csv("NSLKDD/KDDTest+.txt", names=columns)

data_train


# data_train.columns = columns
# data_test.columns = columns

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack,level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20


In [13]:
data_train['attack'].value_counts()

attack
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

Data Normalization

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
std_scaler = StandardScaler()
numeric_col = data_train.select_dtypes(include='number').columns
def normalization(df,col):
  for i in col:
    arr = df[i]
    arr = np.array(arr)
    df[i] = std_scaler.fit_transform(arr.reshape(len(arr),1))
  return df

In [15]:
class IDSModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(IDSModel, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.fc(x)

In [17]:
def keep_numeric_columns_pd(arr):
    df = pd.DataFrame(arr)
    keep = []
    for col in df.columns:
        conv = pd.to_numeric(df[col], errors='coerce')
        # treat a column as numeric if every non-missing original value converted to numeric
        non_numeric = conv.isna() & df[col].notna()
        if not non_numeric.any():
            keep.append(col)
    return df[keep].astype(float).to_numpy(), keep

Binary Classification:

In [20]:
bin_label_train = pd.DataFrame(data_train.attack.map(lambda x:'normal' if x=='normal' else 'abnormal'))
bin_label_test = pd.DataFrame(data_test.attack.map(lambda x:'normal' if x=='normal' else 'abnormal'))

bin_train = data_train.copy()
bin_train['attack'] = bin_label_train
bin_test = data_test.copy()
bin_test['attack'] = bin_label_test

enc_label = preprocessing.LabelEncoder()
bin_train['attack'] = enc_label.fit_transform(bin_train['attack'])

bin_data_train = pd.get_dummies(bin_train,columns=['attack'],prefix="",prefix_sep="") 
bin_data_train['attack'] = bin_label_train
bin_data_test = pd.get_dummies(bin_test,columns=['attack'],prefix="",prefix_sep="")
bin_data_test['attack'] = bin_label_test

bin_data_train = normalization(bin_data_train.copy(),numeric_col)
bin_data_test = normalization(bin_data_test.copy(),numeric_col)
bin_data_train


#bin_data_train = bin_data_train[numeric_col] + bin_data_train['attack']
#bin_data_test = bin_data_test[numeric_col] + bin_data_test['attack']

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,level,0,1,attack
0,-0.110249,tcp,ftp_data,SF,-0.007679,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,0.069972,-0.289103,-0.639532,-0.624871,-0.224532,-0.376387,0.216426,False,True,normal
1,-0.110249,udp,other,SF,-0.007737,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,2.367737,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,-1.965556,False,True,normal
2,-0.110249,tcp,private,S0,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387,-0.219970,True,False,abnormal
3,-0.110249,tcp,http,SF,-0.007723,-0.002891,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.383108,0.066252,-0.572083,-0.602433,-0.387635,-0.345084,0.652823,False,True,normal
4,-0.110249,tcp,http,SF,-0.007728,-0.004814,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.480197,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,0.652823,False,True,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,-0.110249,tcp,private,S0,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387,0.216426,True,False,abnormal
125969,-0.107178,udp,private,SF,-0.007744,-0.004883,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.447834,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387,0.652823,False,True,normal
125970,-0.110249,tcp,smtp,SF,-0.007382,-0.004823,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.480197,-0.289103,0.979238,-0.624871,-0.355014,-0.376387,-0.656367,False,True,normal
125971,-0.110249,tcp,klogin,S0,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387,0.216426,True,False,abnormal


In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# input_dim = data_train.shape[1]
# num_classes = len(np.unique(data_train))
# #model = IDSModel(input_dim, num_classes).to(device)

# #model = NSLKDD_Transformer(num_features=41, num_classes=2).to(device)
# model = IDSModel(input_dim=45, num_classes=2).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# for epoch in range(10):
#     model.train()
#     epoch_loss = 0

#     for data, labels in train_loader:
#         data, labels = data.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(data)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         epoch_loss += loss.item()

#     print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

In [ ]:
X_train = bin_data_train.select_dtypes(include='number').to_numpy()

#X_train = bin_data_train.iloc[:,0:93].to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y_train = bin_data_train['attack'] # target attribute

X_test = bin_data_test.select_dtypes(include='number').to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y_test = bin_data_test['attack'] # target attribute

lsvm = SVC(kernel='linear',gamma='auto') 
lsvm.fit(X_train,y_train) # training model on training dataset

y_pred = lsvm.predict(X_test) # predicting on test dataset
accuracy = accuracy_score(y_test, y_pred)*100
print(f"Accuracy of Linear SVM on NSL-KDD dataset: {accuracy:.2f}%")

Accuracy of Linear SVM on NSL-KDD dataset: 80.76%
